# 📋 Dataset Processing Template

This notebook serves as a **standardized template** for processing all datasets in the Australian ML Datasets repository.

## Guidelines:
Every new dataset should follow this exact structure:

1. **Dataset Information** - Source, context, and metadata
2. **Import Libraries** - All necessary dependencies
3. **Load Raw Data** - Load from `raw/` folder
4. **Exploratory Data Analysis** - Understand the data
5. **Data Cleaning** - Remove duplicates, handle missing values, remove outliers
6. **Feature Engineering** - Create domain-specific features
7. **Prepare for ML** - Train/test split, scaling
8. **Save Datasets** - Save to `processed/` and `features/` folders
9. **Create Metadata** - Document all transformations

## Folder Structure for Each Dataset:
```
datasets/[domain]/[dataset_name]/
├── raw/                          # Original unprocessed data
│   └── [original_file.csv]
├── processed/                    # Cleaned data
│   ├── [name]_clean.csv
│   └── [name]_features.csv
├── features/                     # ML-ready data
│   ├── X_train_scaled.csv
│   ├── X_test_scaled.csv
│   ├── y_train.csv
│   ├── y_test.csv
│   └── scaler.pkl
├── metadata.json                 # Dataset documentation
└── [dataset_name]_pipeline.ipynb # Processing notebook
```

---

## Steps to Add a New Dataset:

### Step 1: Create the folder structure
```bash
# Example: Adding a housing dataset
mkdir -p datasets/housing/new_dataset_name/{raw,processed,features}
```

### Step 2: Copy this notebook to the dataset folder
```bash
cp datasets/housing/new_dataset_name/new_dataset_pipeline.ipynb
```

### Step 3: Edit the notebook
- Replace dataset name and paths
- Adjust cleaning steps for your dataset
- Add domain-specific feature engineering

### Step 4: Run the notebook
- Executes all cells in order
- Generates cleaned, processed, and ML-ready files
- Creates metadata.json automatically

### Step 5: Commit to GitHub
```bash
git add datasets/housing/new_dataset_name/
git commit -m "Add: new_dataset_name from [source]"
git push origin main
```

---

## Key Functions for Dataset Processing

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import pickle
import warnings
warnings.filterwarnings('ignore')

def clean_dataset(df, target_col=None):
    """
    Standardized data cleaning pipeline
    
    Parameters:
    -----------
    df : pd.DataFrame
        Raw dataframe to clean
    target_col : str
        Name of target column (will preserve this column)
    
    Returns:
    --------
    pd.DataFrame
        Cleaned dataframe
    """
    print("🧹 CLEANING DATASET")
    print("=" * 80)
    
    df_clean = df.copy()
    
    # 1. Remove duplicates
    duplicates = df_clean.duplicated().sum()
    df_clean = df_clean.drop_duplicates()
    print(f"✓ Removed {duplicates} duplicate rows")
    
    # 2. Standardize columns
    df_clean.columns = df_clean.columns.str.strip().str.lower().str.replace(' ', '_')
    print(f"✓ Standardized column names")
    
    # 3. Handle missing values
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].median(), inplace=True)
    
    categorical_cols = df_clean.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df_clean[col].isnull().sum() > 0:
            df_clean[col].fillna(df_clean[col].mode()[0], inplace=True)
    print(f"✓ Handled missing values")
    
    # 4. Remove outliers
    for col in numeric_cols:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        df_clean = df_clean[(df_clean[col] >= Q1 - 1.5*IQR) & (df_clean[col] <= Q3 + 1.5*IQR)]
    print(f"✓ Removed outliers using IQR method")
    
    # 5. Encode categorical variables
    label_encoders = {}
    for col in categorical_cols:
        if col not in ['date', 'timestamp']:
            le = LabelEncoder()
            df_clean[col + '_encoded'] = le.fit_transform(df_clean[col].astype(str))
            label_encoders[col] = le
    print(f"✓ Encoded {len(label_encoders)} categorical columns")
    
    print(f"\n✅ Cleaning complete! Final shape: {df_clean.shape}\n")
    return df_clean, label_encoders

def prepare_ml_data(df, target_col, test_size=0.2, random_state=42):
    """
    Prepare data for machine learning
    
    Parameters:
    -----------
    df : pd.DataFrame
        Cleaned dataframe with features
    target_col : str
        Name of target column (encoded)
    test_size : float
        Proportion for test set
    random_state : int
        Random seed
    
    Returns:
    --------
    dict
        Dictionary with train/test sets and scaler
    """
    print("🔧 PREPARING ML DATA")
    print("=" * 80)
    
    # Separate features and target
    y = df[target_col]
    X = df.drop(columns=[target_col] + [col for col in df.columns if col.startswith('date') or col.startswith('location')])
    
    # Keep only numeric features
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    X = X[numeric_cols]
    
    # Train/Test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    # Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"✓ Train/Test split: {len(X_train)} / {len(X_test)}")
    print(f"✓ Features: {len(numeric_cols)}")
    print(f"✓ Data standardized\n")
    
    return {
        'X_train': pd.DataFrame(X_train_scaled, columns=numeric_cols),
        'X_test': pd.DataFrame(X_test_scaled, columns=numeric_cols),
        'y_train': y_train,
        'y_test': y_test,
        'scaler': scaler,
        'feature_names': numeric_cols
    }

def save_dataset(dataset_dict, output_path, dataset_name, dataset_info):
    """
    Save all processed datasets and metadata
    
    Parameters:
    -----------
    dataset_dict : dict
        ML data dictionary from prepare_ml_data()
    output_path : Path
        Output directory path
    dataset_name : str
        Name of the dataset
    dataset_info : dict
        Metadata information
    """
    print("💾 SAVING DATASETS")
    print("=" * 80)
    
    features_path = output_path / 'features'
    features_path.mkdir(exist_ok=True)
    
    # Save ML datasets
    dataset_dict['X_train'].to_csv(features_path / 'X_train_scaled.csv', index=False)
    dataset_dict['X_test'].to_csv(features_path / 'X_test_scaled.csv', index=False)
    dataset_dict['y_train'].to_csv(features_path / 'y_train.csv', index=False, header=['target'])
    dataset_dict['y_test'].to_csv(features_path / 'y_test.csv', index=False, header=['target'])
    
    # Save scaler
    with open(features_path / 'scaler.pkl', 'wb') as f:
        pickle.dump(dataset_dict['scaler'], f)
    
    # Save metadata
    metadata = {
        "dataset_name": dataset_name,
        "created_date": str(datetime.now()),
        "feature_list": dataset_dict['feature_names'],
        "total_features": len(dataset_dict['feature_names']),
        "total_samples": len(dataset_dict['X_train']) + len(dataset_dict['X_test']),
        "train_samples": len(dataset_dict['X_train']),
        "test_samples": len(dataset_dict['X_test']),
        **dataset_info
    }
    
    with open(output_path / 'metadata.json', 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"✓ Saved X_train: {dataset_dict['X_train'].shape}")
    print(f"✓ Saved X_test: {dataset_dict['X_test'].shape}")
    print(f"✓ Saved scaler and metadata\n")
    print(f"✅ All datasets saved to {output_path}\n")

print("✅ Utility functions loaded successfully!")

---

## 📝 Checklist for Adding New Datasets

Use this checklist every time you add a new dataset:

### Before Processing:
- [ ] Download dataset from source
- [ ] Create folder structure: `datasets/[domain]/[dataset_name]/{raw,processed,features}`
- [ ] Place raw CSV/data file in `raw/` folder
- [ ] Copy processing notebook to the dataset folder
- [ ] Update notebook with dataset name and source information

### Processing Steps:
- [ ] Update dataset paths in the notebook
- [ ] Run data loading and exploration cells
- [ ] Review data quality issues
- [ ] Customize data cleaning steps if needed
- [ ] Add domain-specific feature engineering
- [ ] Run train/test split and scaling
- [ ] Verify all output files are created

### Post-Processing:
- [ ] Review metadata.json for accuracy
- [ ] Update data dictionary if needed
- [ ] Test loading from saved files
- [ ] Add README.md to dataset folder (optional)

### Commit to GitHub:
```bash
git add datasets/[domain]/[dataset_name]/
git commit -m "Add: [dataset_name] from [source]"
git push origin main
```

### Example: Adding Housing Dataset
```bash
mkdir -p datasets/housing/sydney_property_prices/{raw,processed,features}
# Place data in: datasets/housing/sydney_property_prices/raw/properties.csv
# Run the processing notebook
# Automatically creates cleaned and ML-ready files
```

---

## File Organization Summary

Every dataset should have:

✅ **raw/** - Original unprocessed data (from source)
✅ **processed/** - Cleaned data + features
✅ **features/** - ML-ready datasets (X_train, X_test, y_train, y_test)
✅ **metadata.json** - Full documentation of transformations
✅ **[name]_pipeline.ipynb** - Processing notebook

This ensures:
- Reproducibility
- Easy to trace transformations
- Ready for immediate ML model training
- Well-documented for future reference